In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import keras
from torch import nn
%matplotlib inline

In [13]:
from keras.datasets import mnist

(xTrain,yTrain), (xTest,yTest) = mnist.load_data() #60k train, 10k test

In [14]:
from sklearn.decomposition import PCA

def flattener(X):
    return np.reshape(X, (len(X), 28*28))
def reshaper(X):
    return np.reshape(X,(len(X), 28,28))

def accuracy(yPred,yTrue):
    nTrue = (yPred==yTrue).int().sum()
    return nTrue/len(yTrue)

# xTrain = torch.FloatTensor(flattener(xTrain))
# xTest = torch.FloatTensor(flattener(xTest))
xTrain = flattener(xTrain)
xTest = flattener(xTest)

pca = PCA(n_components=0.95)
xTrain_red = pca.fit_transform(xTrain) # dims=154 kardega
xTest_red = pca.transform(xTest)

yTrain = torch.LongTensor(yTrain)
yTest = torch.LongTensor(yTest)

xTrain_red = torch.FloatTensor(xTrain_red)
xTest_red = torch.FloatTensor(xTest_red)

xVal = xTest_red[:5000]
xTest_red = xTest_red[5000:]
yVal = yTest[:5000]
yTest = yTest[5000:]

In [15]:
import torch.nn.functional as F

class neuralNetwork(nn.Module):
    def __init__(self,n_dims:int):
        super().__init__()
        self.n_dims = n_dims
        
        self.flatten = nn.Flatten()

        self.input_ = nn.Linear(n_dims,16)
        self.hl1 = nn.Linear(16,32)
        self.hl2=nn.Linear(32,32)
        self.output_ = nn.Linear(32,10)

        self.dropout = nn.Dropout(p=0.3)

    def forward(self,X):
        X = F.relu(self.input_(X)) # Why apply RELU on input layer?? And why is the loss better for Relu over non Relu
        X = F.relu(self.hl1(X))
        X = F.relu(self.hl2(X))
        X = self.dropout(X)
        X = F.softmax(self.output_(X))

        return X

In [16]:
def trainPass(X,y,nEpoch: int,optimizer,model,loss_function, xTest=None, yTest=None, printEpochs=False):
    bestAcc=0
    bestEpoch = 0
    for i in range(nEpoch):
        optimizer.zero_grad()
        
        yPreds = model.forward(X)
        trainLoss = loss_function(yPreds,y)
        trainLoss.backward()
        optimizer.step()

        yPreds = torch.max(yPreds,1).indices
        trainAcc = round(float(accuracy(yPreds,y)),3)
        with torch.no_grad():
            yTestPreds = model.forward(xTest)
            yTestPreds = torch.max(yTestPreds,1).indices
            testAcc = round(float(accuracy(yTestPreds,yTest)),3)

        if(testAcc>=bestAcc and testAcc>=0.97):
            bestAcc=testAcc
            bestEpoch = i
            torch.save(model.state_dict(), f"best_model_params.pt")

        if(printEpochs==True):
            if(yTest!=None and xTest!= None):
                if(i%10==0):
                    print(f"At epoch={i}, Train Acc={trainAcc}, Test Acc={testAcc}")
            else:
                if(i%10==0):
                    print(f"At epoch={i}, Train Acc={trainAcc}")
    if(bestAcc>=0.97):
        print(f"Goal reached. Best acc was {bestAcc} at {bestEpoch}'th epoch")
    

In [17]:
lossFn = nn.CrossEntropyLoss()
torch.manual_seed(42)

model = neuralNetwork(n_dims=154)
model=model.train()
optimizer = torch.optim.NAdam(model.parameters(),lr=0.021, eps=1e-8)
trainPass(xTrain_red,yTrain,400,optimizer,model,lossFn,xTest_red,yTest,printEpochs=True)

/tmp/ipykernel_5568/424482863.py:22: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  X = F.softmax(self.output_(X))


At epoch=0, Train Acc=0.112, Test Acc=0.296
At epoch=10, Train Acc=0.77, Test Acc=0.829
At epoch=20, Train Acc=0.781, Test Acc=0.86
At epoch=30, Train Acc=0.911, Test Acc=0.941
At epoch=40, Train Acc=0.931, Test Acc=0.952
At epoch=50, Train Acc=0.941, Test Acc=0.955
At epoch=60, Train Acc=0.939, Test Acc=0.955
At epoch=70, Train Acc=0.949, Test Acc=0.96
At epoch=80, Train Acc=0.948, Test Acc=0.958
At epoch=90, Train Acc=0.949, Test Acc=0.962
At epoch=100, Train Acc=0.946, Test Acc=0.959
At epoch=110, Train Acc=0.948, Test Acc=0.963
At epoch=120, Train Acc=0.956, Test Acc=0.964
At epoch=130, Train Acc=0.958, Test Acc=0.964
At epoch=140, Train Acc=0.956, Test Acc=0.962
At epoch=150, Train Acc=0.955, Test Acc=0.962
At epoch=160, Train Acc=0.959, Test Acc=0.962
At epoch=170, Train Acc=0.959, Test Acc=0.964
At epoch=180, Train Acc=0.96, Test Acc=0.961
At epoch=190, Train Acc=0.957, Test Acc=0.964
At epoch=200, Train Acc=0.96, Test Acc=0.966
At epoch=210, Train Acc=0.961, Test Acc=0.963
At e

In [19]:
with torch.no_grad():
    model = model.eval()
    evalPreds = model.forward(xVal)
    evalPreds = torch.max(evalPreds,1).indices
    print(accuracy(evalPreds,yVal))

tensor(0.9318)


/tmp/ipykernel_5568/424482863.py:22: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  X = F.softmax(self.output_(X))
